In [1]:
from __future__ import annotations

import glob
from pathlib import Path
import pickle

import torch

from pykeen.models import RotatE

from pykeen_pipeline import load_config, get_dataset


In [2]:
configs = glob.glob("configs/dataset/disease_protein*.yaml")

for dataset_cfg_path in configs:
    print(f"Processing dataset config: {dataset_cfg_path}")
    # dataset_config = load_config(Path(dataset_cfg_path))

    # data_path = Path("dataset_saves")
    # dataset, dataset_label = get_dataset(data_path, dataset_config)


Processing dataset config: configs/dataset\disease_protein_anatomy.yaml
Processing dataset config: configs/dataset\disease_protein_bioprocess.yaml
Processing dataset config: configs/dataset\disease_protein_cellcomp.yaml
Processing dataset config: configs/dataset\disease_protein_drug.yaml
Processing dataset config: configs/dataset\disease_protein_exposure.yaml
Processing dataset config: configs/dataset\disease_protein_hetero.yaml
Processing dataset config: configs/dataset\disease_protein_homo.yaml
Processing dataset config: configs/dataset\disease_protein_molecular.yaml
Processing dataset config: configs/dataset\disease_protein_pathway.yaml
Processing dataset config: configs/dataset\disease_protein_phenotype.yaml


In [20]:
configs = [
    # 'configs/dataset/disease_protein_anatomy.yaml', 
    # 'configs/dataset/disease_protein_bioprocess.yaml', these are done
    'configs/dataset/disease_protein_cellcomp.yaml', 
    # 'configs/dataset/disease_protein_drug.yaml', 
    # 'configs/dataset/disease_protein_exposure.yaml', 
    # 'configs/dataset/disease_protein_hetero.yaml', 
    # 'configs/dataset/disease_protein_homo.yaml', 
    # 'configs/dataset/disease_protein_molecular.yaml', 
    # 'configs/dataset/disease_protein_pathway.yaml', 
    # 'configs/dataset/disease_protein_phenotype.yaml'
    ]

config = configs[0]
print(f"Processing dataset config: {config}")
dataset_config = load_config(Path(config))

data_path = Path("dataset_saves")
dataset, dataset_label = get_dataset(data_path, dataset_config)

Processing dataset config: configs/dataset/disease_protein_cellcomp.yaml
Loading dataset from dataset_saves\disease_protein_cellcomp.pkl
Train: 503927, Validation: 2000, Test: 16000
Inverse relations - Train: True, Validation: False, Test: False


In [18]:
dataset.testing.relation_labeling

Labeling(label_to_id={'cellcomp_cellcomp': 0, 'cellcomp_protein': 1, 'disease_disease': 2, 'disease_protein': 3, 'protein_protein': 4}, id_to_label={0: 'cellcomp_cellcomp', 1: 'cellcomp_protein', 2: 'disease_disease', 3: 'disease_protein', 4: 'protein_protein'}, _vectorized_mapper=<numpy.vectorize object at 0x0000020F0298DED0>, _vectorized_labeler=<numpy.vectorize object at 0x0000020F0298E050>)

In [2]:
dataset_name = "disease_protein_anatomy"
dataset_path = f"dataset_saves/{dataset_name}.pkl"
model_path =  Path(glob.glob(f'dataset_gridsearch_results/{dataset_name}_*')[0])
run_config_path =model_path / "config.yaml"
# configból a disease_protein_* és azt betölteni
# models = glob.glob("dataset_gridsearch_results/disease_protein*")
# print(models)
# model_path = f"{model_path}/trained_model.pkl"
model_path = model_path / "best_model.pth"
print(model_path, dataset_path)

# model = torch.load(Path(model_path), map_location="cpu")

if Path(dataset_path).exists():
    print(f"Loading dataset from {dataset_path}")
    with Path(dataset_path).open("rb") as f:
        dataset = pickle.load(f)
        dataset = dataset.get_dataset()

# ezekben vannak az egyes futások
# trained_model.pkl és checkpoint.pth
run_config = load_config(run_config_path)
model = RotatE(
    triples_factory=dataset.training,
    **run_config.get("model_kwargs", {}),
)
model.load_state_dict(torch.load(Path(model_path), map_location="cpu"))


dataset_gridsearch_results\disease_protein_anatomy_RotatE_09c21bfc\best_model.pth dataset_saves/disease_protein_anatomy.pkl
Loading dataset from dataset_saves/disease_protein_anatomy.pkl
Train: 1967802, Validation: 2000, Test: 16000
Inverse relations - Train: True, Validation: False, Test: False


ConstructorError: could not determine a constructor for the tag 'tag:yaml.org,2002:python/object/apply:pathlib.WindowsPath'
  in "<unicode string>", line 37, column 20:
      best_model_path: !!python/object/apply:pathlib.Wi ... 
                       ^

In [14]:
from pykeen.evaluation import RankBasedEvaluator


dataset_name = "disease_protein_anatomy"
dataset_path = f"dataset_saves/{dataset_name}.pkl"
run_dir = Path(glob.glob(f'dataset_gridsearch_results/{dataset_name}_*')[0])
run_config = load_config(run_dir / "config.yaml")
model_path = run_dir / "best_model.pth"

if Path(dataset_path).exists():
    print(f"Loading dataset from {dataset_path}")
    with Path(dataset_path).open("rb") as f:
        dataset = pickle.load(f)
        dataset = dataset.get_dataset()

model = RotatE(
    triples_factory=dataset.training,
    **run_config.get("model_kwargs", {}),
)
model.load_state_dict(torch.load(model_path, map_location="cpu"))


# Define evaluator
evaluator = RankBasedEvaluator(
    filtered=True,  # Note: this is True by default; we're just being explicit
)

# Evaluate your model with not only testing triples,
# but also filter on validation triples
results = evaluator.evaluate(
    model=model,
    mapped_triples=dataset.testing.mapped_triples,
    additional_filter_triples=[
        dataset.training.mapped_triples,
        dataset.validation.mapped_triples,
    ],
    device="cpu"
)


ConstructorError: could not determine a constructor for the tag 'tag:yaml.org,2002:python/object/apply:pathlib.WindowsPath'
  in "<unicode string>", line 37, column 20:
      best_model_path: !!python/object/apply:pathlib.Wi ... 
                       ^